<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/Ultralytics/wl-how-to-train-ultralytics-yolo-on-brain-tumor-detection-dataset.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Train an <a href="https://github.com/ultralytics/ultralytics">Ultralytics YOLO</a> detector on the MRI <b>brain-tumor</b> dataset and stream every sample, prediction and metric live into <a href="https://github.com/GrayboxTech/weightslab">Weights Studio</a> to <b>inspect, edit and evolve</b> your run.</div>

# Brain Tumor Detection (MRI) · Ultralytics YOLO + WeightsLab

This notebook trains a YOLO detector on the [brain-tumor](https://docs.ultralytics.com/datasets/detect/brain-tumor/) MRI dataset **through the WeightsLab training pipeline**.

Instead of Ultralytics' one-shot `train`/`predict` flow, we hand training to `WLAwareTrainer`. It wraps the train/val dataloaders, logs a **per-sample** signal for every image (loss, prediction stats, tags) and ships live prediction overlays to **Weights Studio** — where you inspect the data grid, remove/weight samples, edit the model and resume, all while training runs.

**Dataset** — 893 training images and 223 validation images, each with detection annotations. Classes: `0: negative`, `1: positive`. Ultralytics auto-downloads it (~4 MB) the first time you train.

## 1 · Setup

Install WeightsLab (which pulls in Ultralytics) and verify the environment.

In [ ]:
%pip install weightslab ultralytics

import ultralytics
ultralytics.checks()

import weightslab as wl

## 2 · Dataset

The dataset is described by a small YAML file that Ultralytics resolves and downloads automatically — no manual setup needed. For reference, `brain-tumor.yaml` looks like this:

```yaml
path: brain-tumor      # dataset root dir (auto-downloaded)
train: images/train    # 893 images
val: images/val        # 223 images
names:
  0: negative
  1: positive
```

To train on **your own** data, point `data` (in the next cell) at your own `data.yaml` in YOLO format.

## 3 · Train through WeightsLab

We register the run's hyperparameters with `wl.watch_or_edit(...)` (so they become live-editable from Studio), start the backend services with `wl.serve()`, then hand training to `WLAwareTrainer` via the standard `YOLO(...).train(trainer=...)` entry point.

> **Connect Weights Studio** — run `weightslab ui launch` locally (opens `http://localhost:5173`) *before or during* training to watch the run live. On **Colab**, serve over a public tunnel instead: use `wl.serve(serving_grpc=True, serving_bore=True)` and connect your local Studio with `weightslab tunnel bore.pub:PORT` (the port is printed when serving starts).

Data augmentations are disabled so each sample maps cleanly to its ground truth in the grid.

In [ ]:
import os
os.environ.setdefault("WL_PRELOAD_IMAGE_OVERVIEW", "0")
os.environ.setdefault("WEIGHTSLAB_LOG_LEVEL", "WARNING")

import warnings
warnings.filterwarnings("ignore")

from weightslab.integrations.ultralytics import WLAwareTrainer
from ultralytics import YOLO

# Hyperparameters registered with WeightsLab -> live-editable from Weights Studio.
cfg = {
    "model": "yolo11n.pt",       # pretrained checkpoint
    "data": "brain-tumor.yaml",  # Ultralytics auto-downloads (~4 MB)
    "image_size": 640,
    "epochs": 10,
    # Per-sample prediction overlays streamed to Studio (NMS on the train set).
    "signals_cfg": {
        "train_nms": {"conf_thres": 0.25, "iou_thres": 0.45, "max_nms": 7},
    },
}
wl.watch_or_edit(cfg, flag="hyperparameters", defaults=cfg, poll_interval=1.0)

# Start the WeightsLab backend services that Weights Studio connects to.
wl.serve(serving_grpc=True)
#   Colab -> local Studio:  wl.serve(serving_grpc=True, serving_bore=True)

wl.start_training()  # equivalent to pressing "play" in Studio

# Read back the (now live) hyperparameters and hand training to WLAwareTrainer.
model = YOLO(cfg["model"])
results = model.train(
    trainer=WLAwareTrainer,
    data=str(cfg["data"]),
    imgsz=cfg["image_size"],
    epochs=int(cfg["epochs"]),
    project="./wl_logs", name="brain-tumor",  # -> WL log_dir/name
    workers=0,          # WL invariant (parent-process uid counter)
    optimizer="SGD", lr0=0.001, amp=False,
    # All augmentations off for a clean sample <-> ground-truth association.
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0, erasing=0.0, auto_augment=None,
)

## 4 · Explore the data grid

The table below is exactly the data behind Weights Studio's **grid explorer**: one row per sample, keyed by split (`origin`) and sample id, with the per-sample loss, prediction stats and tags accumulated during training. This is the ledger that Studio reads from — here we render it inline with pandas.

The image thumbnails and bounding-box overlays themselves are rendered by **Weights Studio** (which streams full previews over gRPC); the link below opens the interactive grid there.

In [ ]:
import pandas as pd
from IPython.display import display, HTML
from weightslab.backend.ledgers import get_dataframe

# `get_df_view()` is the same per-sample DataFrame that powers Studio's grid.
dfm = get_dataframe()
grid = dfm.get_df_view() if hasattr(dfm, "get_df_view") else pd.DataFrame()
print(f"{len(grid)} samples tracked")
display(grid.head(50))

# Open the live, interactive grid (with image + bbox previews) in Weights Studio.
# Studio must be running locally (`weightslab ui launch`).
display(HTML(
    '<a href="http://localhost:5173" target="_blank" '
    'style="font-size:15px">&#128279; Open the full grid in Weights Studio</a>'
))

> **Can clicking a grid row here jump to that sample in Studio?** Not out of the box today. The notebook and Studio are separate processes, and the link above opens Studio at its root. Per-sample deep-linking (e.g. `http://localhost:5173/?sample=<uid>` selecting and scrolling to that row) is a small frontend addition to Weights Studio — the sample `uid` is already the grid's index here, so the notebook side is ready for it once Studio accepts the query param.

## 5 · Export (optional)

The best checkpoint is saved under the run's `weights/` directory. Export it to any deployment format with the `format` argument (e.g. `onnx`, `engine`, `openvino`).

In [ ]:
from ultralytics import YOLO

best = f"{model.trainer.save_dir}/weights/best.pt"  # run dir chosen by Ultralytics
YOLO(best).export(format="onnx")

## Citation

```bibtex
@dataset{Jocher_Ultralytics_Datasets_2024,
    author = {Jocher, Glenn and Rizwan, Muhammad},
    license = {AGPL-3.0},
    title = {Ultralytics Datasets: Brain-tumor Detection Dataset},
    url = {https://docs.ultralytics.com/datasets/detect/brain-tumor/},
    version = {1.0.0},
    year = {2024}
}
```